In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *


In [3]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [4]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.M_720s.20241120_001200_TAI.3.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.V_45s.20241120_001500_TAI.2.Dopplergram.fits',
 '/home/ulyanov/data/cross/hmi.V_720s.20241120_001200_TAI.3.Dopplergram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20241120T001503_V202602220151_0451200501.fits.gz']

In [5]:
file_hmi = files[0]
file_fdt = files[4]

file_hmi, file_fdt

('/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz')

In [6]:
for i, dcrota in enumerate(np.arange(0.28,0.32,0.001)):


    with fits.open(file_hmi) as hdul:
        header_hmi = hdul[1].header
        data_hmi = hdul[1].data

    with fits.open(file_fdt) as hdul:
        header_fdt = hdul[0].header
        data_fdt = hdul[0].data

    data_fdt = undistort(data_fdt, header_fdt, xd, yd)

    data_hmi = rebin(data_hmi, 4, update_header=header_hmi)
    data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=True, mu_thr=0.5, crota=header_fdt['CROTA'] + dcrota, rsun=330.25, xc=421.64, yc=457.87)

    data_fdt[np.isnan(data_hmi)] = np.nan

    q = np.nanmean(data_fdt * data_hmi) / np.sqrt(np.nanmean(data_fdt ** 2) * np.nanmean(data_hmi ** 2))
    #cor[i] = q

    print(dcrota,  q)

0.28 0.9005426864446505
0.281 0.9005623877234928
0.28200000000000003 0.900579598532147
0.28300000000000003 0.9005934536400536
0.28400000000000003 0.9006036965743706
0.28500000000000003 0.9006096661134458
0.28600000000000003 0.9006126278844782
0.28700000000000003 0.9006134891708636
0.28800000000000003 0.9006135628691564
0.28900000000000003 0.900611555392505
0.29000000000000004 0.9006057157861932
0.29100000000000004 0.9005972319028466
0.29200000000000004 0.9005855833163191
0.29300000000000004 0.9005734625714491
0.29400000000000004 0.9005586555719562
0.29500000000000004 0.9005421981565189
0.29600000000000004 0.9005216758310376
0.29700000000000004 0.9004979307499709
0.29800000000000004 0.9004727814876028
0.29900000000000004 0.9004449572444619
0.30000000000000004 0.900414635453733
0.30100000000000005 0.9003830292215415
0.30200000000000005 0.9003484992806722
0.30300000000000005 0.9003107785121274
0.30400000000000005 0.9002695140377054
0.30500000000000005 0.9002257365669308
0.3060000000000000